# 03 · From pixels to a sequence: backbone, projection, masks, positions

> **Paper:** §3.2 "Backbone" and "Transformer encoder" · **Code:** [`models/backbone.py`](../models/backbone.py), [`models/position_encoding.py`](../models/position_encoding.py)

A transformer eats a **sequence** of vectors. A photo is a **2-D grid** of pixels. This notebook is the bridge, and it is almost entirely about **tensor shapes**.

We'll follow one image through four transformations:

```
(3, 800, 1066)   PIL image -> tensor                    3 channels, H, W
(2048, 25, 34)   ResNet-50                              lots of channels, tiny grid
( 256, 25, 34)   1x1 conv "input_proj"                  squeeze channels to d=256
( 850, 256)      flatten H,W into one sequence axis     850 = 25*34 "pixels" as tokens
```

Plus two things that trip everyone up: the **padding mask** and the **positional encoding**.

> **Shape vocabulary.** This notebook leans hard on `flatten`, `permute`, `view` and broadcasting. If any of those are shaky, [`00 · PyTorch essentials`](00_pytorch_essentials.ipynb) §2–§3 takes them apart one verb at a time, using these exact tensors.

## 0. Setup — everything this notebook needs

This notebook is **self-contained**: only third-party packages are imported, and every DETR-specific piece is written out below. Nothing comes from this repo, so you can read straight through without chasing a helper into another file.

Run this section once, then forget about it.

In [ ]:
# Standard third-party imports. Nothing from this repo -- every helper this
# notebook uses is defined below, in this file.
import itertools
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=110)
np.set_printoptions(precision=3, suppress=True)

ASSETS = "_assets"                      # downloaded images are cached here
os.makedirs(ASSETS, exist_ok=True)
print("torch", torch.__version__)

### COCO class names and plot colors

In [ ]:
# DETR predicts 91 "classes" + 1 no-object slot = 92 logits per query.
# COCO's category ids are not contiguous (they run 1..90 with gaps), so the gaps
# are filled with 'N/A' placeholders and index 0 is unused. The list MUST be
# exactly 91 long -- one short and every label after the gap is silently wrong.
COCO_CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush',
]
assert len(COCO_CLASSES) == 91, f"expected 91 classes, got {len(COCO_CLASSES)}"

COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

### Box geometry

DETR predicts boxes as **`cxcywh`** — centre + size, normalized to `[0, 1]`. IoU and plotting want **`xyxy`** corners. Mixing the two up is the single most common bug in detection code, so both conversions live here.

In [ ]:
def box_cxcywh_to_xyxy(b):
    """(cx, cy, w, h) -> (x0, y0, x1, y1), on the last dim."""
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)


def box_xyxy_to_cxcywh(b):
    x0, y0, x1, y1 = b.unbind(-1)
    return torch.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], dim=-1)


def box_area(b):
    """Area of (x0, y0, x1, y1) boxes."""
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    """Pairwise IoU. a: (N, 4), b: (M, 4), both xyxy. Returns (iou, union), each (N, M)."""
    area_a, area_b = box_area(a), box_area(b)
    lt = torch.max(a[:, None, :2], b[None, :, :2])          # (N, M, 2) top-left of overlap
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])          # (N, M, 2) bottom-right
    wh = (rb - lt).clamp(min=0)                             # no overlap -> 0
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union, union


def generalized_box_iou(a, b):
    """GIoU = IoU - |C \\ (A u B)| / |C|, where C is the smallest box enclosing both.

    Unlike IoU, GIoU keeps giving gradient when the boxes do not overlap at all:
    it measures how far apart they are, in units of the enclosing box.
    Range is [-1, 1] (1 = identical, -1 = infinitely far apart).
    """
    assert (a[:, 2:] >= a[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    assert (b[:, 2:] >= b[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    iou, union = box_iou(a, b)
    lt = torch.min(a[:, None, :2], b[None, :, :2])          # enclosing box
    rb = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    enclosing = wh[:, :, 0] * wh[:, :, 1]
    return iou - (enclosing - union) / enclosing


# --- quick self-check -------------------------------------------------------
_a = torch.tensor([[0.0, 0.0, 2.0, 2.0]])
_b = torch.tensor([[1.0, 1.0, 3.0, 3.0]])
assert torch.allclose(box_iou(_a, _b)[0], torch.tensor([[1 / 7]]))        # 1 / (4+4-1)
assert torch.allclose(generalized_box_iou(_a, _a), torch.tensor([[1.0]])) # identical -> 1
_far = torch.tensor([[10.0, 10.0, 11.0, 11.0]])
assert box_iou(_a, _far)[0].item() == 0.0                                 # IoU dies...
assert generalized_box_iou(_a, _far).item() < 0                           # ...GIoU still ranks
print("box helpers ok")

### Images in, tensors out

DETR's eval transform resizes the shortest side to 800px and ImageNet-normalizes. There is no fixed crop — the model accepts any input size.

In [ ]:
SAMPLE_IMAGES = {
    "cats":    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "street":  "http://images.cocodataset.org/val2017/000000000139.jpg",
    "horses":  "http://images.cocodataset.org/val2017/000000006471.jpg",
    "kitchen": "http://images.cocodataset.org/val2017/000000002153.jpg",
}


def load_image(name_or_url):
    """Load a sample image by nickname, URL, or local path. Cached under _assets/."""
    url = SAMPLE_IMAGES.get(name_or_url, name_or_url)
    if os.path.exists(url):
        return Image.open(url).convert("RGB")
    path = os.path.join(ASSETS, os.path.basename(url))
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.write(requests.get(url, timeout=60).content)
    return Image.open(path).convert("RGB")


# DETR's eval transform: resize the shortest side to 800px, to tensor, ImageNet
# normalize. There is NO fixed crop -- DETR accepts variable input sizes.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
default_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(800),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def get_device():
    """CUDA > MPS (Apple Silicon) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

### Drawing detections

In [ ]:
def rescale_bboxes(boxes, size):
    """Normalized cxcywh in [0,1] -> absolute xyxy pixels. `size` is PIL's (W, H)."""
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(boxes)
    return b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)


def plot_results(pil_img, prob, boxes, ax=None, title=None, linewidth=2.5):
    """prob: (n, 91) softmax WITHOUT the no-object column. boxes: (n, 4) xyxy pixels."""
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(pil_img)
    for i, (p, (xmin, ymin, xmax, ymax)) in enumerate(zip(prob, boxes.tolist())):
        c = COLORS[i % len(COLORS)]
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=linewidth))
        cl = p.argmax()
        ax.text(xmin, ymin, f'{COCO_CLASSES[cl]}: {p[cl]:0.2f}', fontsize=11,
                bbox=dict(facecolor=c, alpha=0.6, edgecolor='none'), color='white')
    ax.axis('off')
    if title:
        ax.set_title(title)
    return ax

### DETR itself

Backbone, positional encoding, transformer and prediction heads, written out. This is the same architecture as [`models/`](../models) with the inference path kept.

The proof that it is faithful is `load_state_dict(...)` below: it is **strict**, so every parameter name here has to match Facebook's released checkpoint exactly or it raises.

In [ ]:
class FrozenBatchNorm2d(nn.Module):
    """BatchNorm with statistics and affine parameters frozen as plain buffers."""
    def __init__(self, n):
        super().__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, *a, **kw):
        state_dict.pop(prefix + "num_batches_tracked", None)
        super()._load_from_state_dict(state_dict, prefix, *a, **kw)

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        scale = w * (rv + 1e-5).rsqrt()
        return x * scale + (b - rm * scale)


class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=128, temperature=10000, scale=2 * math.pi):
        super().__init__()
        self.num_pos_feats, self.temperature, self.scale = num_pos_feats, temperature, scale

    def forward(self, x, mask):
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        eps = 1e-6
        y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
        x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[..., 0::2].sin(), pos_x[..., 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[..., 0::2].sin(), pos_y[..., 1::2].cos()), dim=4).flatten(3)
        return torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)


class Backbone(nn.Module):
    """ResNet-50 trunk with frozen BN, returning only the last (stride-32) stage."""
    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet50(weights=None, norm_layer=FrozenBatchNorm2d)
        self.body = torchvision.models._utils.IntermediateLayerGetter(net, {"layer4": "0"})
        self.num_channels = 2048

    def forward(self, x, mask):
        feat = self.body(x)["0"]
        feat_mask = F.interpolate(mask[None].float(), size=feat.shape[-2:]).to(torch.bool)[0]
        return feat, feat_mask


class TransformerEncoderLayer(nn.Module):
    """One encoder block: self-attention over image tokens, then a feed-forward net."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None, pos=None):
        # pos is added to the QUERY and the KEY but never to the VALUE:
        # position decides where to look, not what gets carried back.
        q = k = src if pos is None else src + pos
        src2 = self.self_attn(q, k, value=src, key_padding_mask=src_key_padding_mask)[0]
        src = self.norm1(src + self.dropout1(src2))
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        return self.norm2(src + self.dropout2(src2))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])

    def forward(self, src, src_key_padding_mask=None, pos=None):
        out = src
        for layer in self.layers:
            out = layer(out, src_key_padding_mask=src_key_padding_mask, pos=pos)
        return out


class TransformerDecoderLayer(nn.Module):
    """One decoder block: queries talk to each other, then to the image, then FFN."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        q = k = tgt if query_pos is None else tgt + query_pos
        tgt2 = self.self_attn(q, k, value=tgt)[0]              # queries deduplicate here
        tgt = self.norm1(tgt + self.dropout1(tgt2))
        tgt2 = self.multihead_attn(                            # queries read the image here
            query=tgt if query_pos is None else tgt + query_pos,
            key=memory if pos is None else memory + pos,
            value=memory, key_padding_mask=memory_key_padding_mask)[0]
        tgt = self.norm2(tgt + self.dropout2(tgt2))
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        return self.norm3(tgt + self.dropout3(tgt2))


class TransformerDecoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        """Returns EVERY layer's output, stacked: (num_layers, num_queries, B, d_model).

        DETR keeps them all because the loss is applied after each decoder layer
        ("auxiliary decoding losses", paper section 3.2).
        """
        out = tgt
        intermediate = []
        for layer in self.layers:
            out = layer(out, memory, memory_key_padding_mask=memory_key_padding_mask,
                        pos=pos, query_pos=query_pos)
            intermediate.append(self.norm(out))
        return torch.stack(intermediate)


class Transformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.0):
        super().__init__()
        self.encoder = TransformerEncoder(d_model, nhead, dim_feedforward,
                                          num_encoder_layers, dropout)
        self.decoder = TransformerDecoder(d_model, nhead, dim_feedforward,
                                          num_decoder_layers, dropout)
        self.d_model, self.nhead = d_model, nhead

    def forward(self, src, mask, query_embed, pos_embed):
        """src/pos_embed: (B, C, H, W). mask: (B, H, W), True = padding."""
        bs, c, h, w = src.shape
        # (B, C, H, W) -> (H*W, B, C): this implementation puts the sequence axis first.
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
        mask = mask.flatten(1)

        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)
        tgt = torch.zeros_like(query_embed)                    # queries start at zero
        hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                          pos=pos_embed, query_pos=query_embed)
        return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(nn.Linear(n, k)
                                    for n, k in zip([input_dim] + h, h + [output_dim]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < len(self.layers) - 1 else layer(x)
        return x


class DETR(nn.Module):
    def __init__(self, num_classes=91, num_queries=100, hidden_dim=256, nheads=8,
                 enc_layers=6, dec_layers=6, dim_feedforward=2048, aux_loss=False):
        super().__init__()
        self.backbone = nn.ModuleList([Backbone(), PositionEmbeddingSine(hidden_dim // 2)])
        self.transformer = Transformer(hidden_dim, nheads, enc_layers, dec_layers,
                                       dim_feedforward)
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        self.query_embed = nn.Embedding(num_queries, hidden_dim)
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)   # +1 = "no object"
        self.bbox_embed = MLP(hidden_dim, hidden_dim, 4, 3)
        self.num_queries = num_queries
        self.aux_loss = aux_loss                # keep every decoder layer's prediction

    def forward(self, images, mask=None):
        """images: (B, 3, H, W). mask: (B, H, W) with True on padded pixels."""
        if mask is None:                        # a single image needs no padding
            mask = torch.zeros(images.shape[0], *images.shape[-2:],
                               dtype=torch.bool, device=images.device)
        feat, feat_mask = self.backbone[0](images, mask)
        pos = self.backbone[1](feat, feat_mask)
        # hs: (num_decoder_layers, B, num_queries, hidden_dim)
        hs, memory = self.transformer(self.input_proj(feat), feat_mask,
                                      self.query_embed.weight, pos)

        outputs_class = self.class_embed(hs)             # (layers, B, queries, classes+1)
        outputs_coord = self.bbox_embed(hs).sigmoid()    # (layers, B, queries, 4)
        out = {"pred_logits": outputs_class[-1], "pred_boxes": outputs_coord[-1]}
        if self.aux_loss:
            out["aux_outputs"] = [{"pred_logits": a, "pred_boxes": b}
                                  for a, b in zip(outputs_class[:-1], outputs_coord[:-1])]
        return out


DETR_R50_URL = "https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth"


def load_pretrained_detr(device=None, aux_loss=False):
    """Build the model above and load Facebook's released COCO weights into it.

    `load_state_dict` is strict by default, which is the real test: every parameter
    name defined above has to match the official checkpoint exactly, or this raises.
    """
    model = DETR(aux_loss=aux_loss)
    ck = torch.hub.load_state_dict_from_url(DETR_R50_URL, map_location="cpu")
    model.load_state_dict(ck["model"])
    model.eval()
    return model.to(device) if device is not None else model


@torch.no_grad()
def detect(model, pil_img, threshold=0.9, device=None):
    """Returns (probs_kept (n, 91), boxes_kept xyxy pixels, raw outputs, keep mask)."""
    device = device or next(model.parameters()).device
    x = default_transform(pil_img).unsqueeze(0).to(device)
    outputs = model(x)
    probs = outputs["pred_logits"].softmax(-1)[0, :, :-1].cpu()   # drop no-object column
    keep = probs.max(-1).values > threshold
    boxes = rescale_bboxes(outputs["pred_boxes"][0, keep].cpu(), pil_img.size)
    return probs[keep], boxes, outputs, keep

In [ ]:
device = get_device()
model = load_pretrained_detr(device=device)
im = load_image("cats")
print("device:", device, "| image (W,H):", im.size)

## 1. Preprocessing: why there is no fixed input size

Recall the cats-and-dogs classifier, where every image was forced to 150×150. Detection can't do that — squashing an image distorts the boxes you're trying to predict.

DETR instead resizes the **shortest side to 800px** and lets the other side land wherever it lands.

In [ ]:
x = default_transform(im)               # shape: (3, H, W)   -- Resize(800) + ToTensor + Normalize
print("PIL  (W, H)      :", im.size)
print("tensor (C, H, W) :", tuple(x.shape))
print("  -> shortest side is exactly 800:", min(x.shape[1:]) == 800)
print("  -> aspect ratio preserved      :",
      round(im.size[0]/im.size[1], 3), "vs", round(x.shape[2]/x.shape[1], 3))

# Different image -> different tensor shape. That is allowed here.
im2 = load_image("street")
print("\nanother image:", im2.size, "->", tuple(default_transform(im2).shape))

### The batch problem, and `NestedTensor`

Two images of different shapes cannot be `torch.stack`ed. DETR's fix is to pad every image in a batch up to the largest H and W, and carry a **boolean mask** marking which pixels are real padding.

That pair — `(padded_tensor, mask)` — is what the repo calls a `NestedTensor` ([`util/misc.py`](../util/misc.py)).

In [ ]:
class NestedTensor:
    """A padded batch of images plus the mask saying which pixels are real.

    Images in a batch have different sizes, so they are padded to a common
    (maxH, maxW). `mask[b, y, x]` is True where pixel (y, x) of image b is padding.
    """
    def __init__(self, tensors, mask):
        self.tensors, self.mask = tensors, mask

    def decompose(self):
        return self.tensors, self.mask

    def to(self, device):
        return NestedTensor(self.tensors.to(device), self.mask.to(device))


def nested_tensor_from_tensor_list(tensor_list):
    """Pad a list of (3, H_i, W_i) tensors into one batch + its padding mask."""
    max_h = max(t.shape[1] for t in tensor_list)
    max_w = max(t.shape[2] for t in tensor_list)
    b = len(tensor_list)
    dtype, device = tensor_list[0].dtype, tensor_list[0].device

    batch = torch.zeros((b, 3, max_h, max_w), dtype=dtype, device=device)
    mask = torch.ones((b, max_h, max_w), dtype=torch.bool, device=device)  # True = padding
    for img, pad_img, m in zip(tensor_list, batch, mask):
        pad_img[:, :img.shape[1], :img.shape[2]].copy_(img)
        m[:img.shape[1], :img.shape[2]] = False                            # real pixels
    return NestedTensor(batch, mask)


im2 = load_image("street")
batch = nested_tensor_from_tensor_list([default_transform(im), default_transform(im2)])
print("tensors shape:", tuple(batch.tensors.shape), "  <- (B, 3, maxH, maxW), both padded")
print("mask    shape:", tuple(batch.mask.shape),    "  <- (B, maxH, maxW), True = padding")
print()
for i in range(2):
    real = (~batch.mask[i]).sum().item()
    total = batch.mask[i].numel()
    print(f"  image {i}: {real:,} real pixels / {total:,} slots  ({100*real/total:.1f}% real)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for i, ax in enumerate(axes):
    ax.imshow(batch.mask[i].cpu(), cmap="gray")
    ax.set_title(f"image {i} padding mask  (white = padded)")
    ax.axis("off")
plt.tight_layout(); plt.show()

**Why the mask matters.** Self-attention is global: without a mask, every real pixel would attend to the grey padding as if it were image content. The mask is threaded all the way through to `nn.MultiheadAttention(key_padding_mask=...)` in [`models/transformer.py:56`](../models/transformer.py#L56).

From here on we use a **single image**, so `B=1` and the mask is all-`False` (nothing padded).

## 2. The backbone: trading resolution for semantics

ResNet-50, with its final classification head removed. It is extracted with `IntermediateLayerGetter(backbone, return_layers={'layer4': '0'})` — i.e. stop at `layer4` and hand back that feature map.

In [ ]:
single = nested_tensor_from_tensor_list([default_transform(im)]).to(device)
print("input :", tuple(single.tensors.shape))

with torch.no_grad():
    # model.backbone[0] is the ResNet trunk, model.backbone[1] the positional encoding
    src, mask = model.backbone[0](single.tensors, single.mask)

print("output:", tuple(src.shape), "  <- (B, C, H/32, W/32)")
print()
H0, W0 = single.tensors.shape[-2:]
C, H, W = src.shape[1:]
print(f"  spatial : {H0}x{W0}  ->  {H}x{W}      (divided by {H0//H}, the ResNet stride)")
print(f"  channels: 3          ->  {C}")
print(f"  so one 'pixel' of the feature map summarizes a {H0//H}x{W0//W} patch of the photo")
print(f"  the mask came along for the ride: {tuple(single.mask.shape)} -> {tuple(mask.shape)}")

This is the classic CNN trade: **spatial resolution down 32×, channel depth up 683×**. Each of the 2048 channels is a learned feature detector; each grid cell now describes a 32×32 patch of the original photo.

Matching the paper (§3.2): *"a conventional CNN backbone generates a lower-resolution activation map f ∈ ℝ^(C×H×W). Typical values we use are C = 2048 and H, W = H₀/32, W₀/32."*

### Frozen BatchNorm — a detail worth knowing

Detection trains with tiny batches (2 images per GPU). BatchNorm statistics computed over 2 images are garbage, so DETR replaces every `BatchNorm2d` with `FrozenBatchNorm2d`: the running mean/var from ImageNet pretraining are baked in as constants.

In [ ]:
bn = [m for m in model.backbone.modules() if isinstance(m, FrozenBatchNorm2d)]
print(f"FrozenBatchNorm2d layers : {len(bn)}")
print(f"trainable BatchNorm2d    : {sum(1 for m in model.backbone.modules() if isinstance(m, torch.nn.BatchNorm2d))}")
print()
print("FrozenBatchNorm2d stores weight/bias/mean/var as *buffers*, not parameters:")
print("   buffers   :", len(list(bn[0].buffers())), "  parameters:", len(list(bn[0].parameters())))
print("   -> nothing to learn, nothing to update")
print()
print("Why: DETR trains with 2 images per GPU. Batch statistics over 2 images are")
print("noise, so the ImageNet statistics are frozen and reused instead.")

## 3. `input_proj`: 2048 → 256 with a 1×1 convolution

A transformer with `d_model=2048` would be enormous. DETR shrinks the channel dimension to **d = 256** first, with a 1×1 convolution.

A 1×1 conv is worth pausing on if you're new to CNNs: it touches **one pixel at a time** and only mixes *channels*. It is exactly an `nn.Linear(2048, 256)` applied independently at every grid location — the same "one weight matrix, applied per position" idea as `nn.Linear` over a batch.

In [ ]:
print("input_proj:", model.input_proj)

with torch.no_grad():
    proj = model.input_proj(src)      # shape: (B, 256, H, W)

print("\nbefore:", tuple(src.shape))
print("after :", tuple(proj.shape), "  <- H and W untouched, only C changed")
print()
w = model.input_proj.weight
print("weight shape:", tuple(w.shape), " = (out_ch, in_ch, 1, 1)")
print("params      :", w.numel() + model.input_proj.bias.numel(),
      "= 2048*256 + 256, identical to nn.Linear(2048, 256)")

## 4. Flatten the grid into a sequence

Now the actual pixels→tokens step. It is one line in [`models/transformer.py:50`](../models/transformer.py#L50):

```python
src = src.flatten(2).permute(2, 0, 1)   # (B, C, H, W) -> (HW, B, C)
```

Read it in two moves:
- `.flatten(2)` collapses dims 2 onward: `(B, 256, 25, 34)` → `(B, 256, 850)`
- `.permute(2, 0, 1)` reorders to `(850, B, 256)`

**850 = 25 × 34.** Each grid cell becomes one token in a sequence of 850. This DETR predates `batch_first=True`, so it uses the older `(sequence, batch, channels)` convention.

In [ ]:
B_, C, H, W = proj.shape
seq = proj.flatten(2).permute(2, 0, 1)      # shape: (H*W, B, C)

print(f"(B, C, H, W)  = {tuple(proj.shape)}")
print(f"(H*W, B, C)   = {tuple(seq.shape)}     <- {H} * {W} = {H*W} tokens")
print()
print("sequence length is NOT fixed -- it depends on the image:")
for name in ["cats", "street", "horses"]:
    t = default_transform(load_image(name))
    # CAREFUL: ResNet rounds UP, not down. 1066/32 = 33.3 -> 34, not 33.
    # Use ceil, or you will be off by a whole row/column of tokens.
    h, w = math.ceil(t.shape[1] / 32), math.ceil(t.shape[2] / 32)
    with torch.no_grad():                      # verify against the real backbone
        nt = nested_tensor_from_tensor_list([t]).to(device)
        real = model.backbone[0](nt.tensors, nt.mask)[0].shape[-2:]
    ok = "OK" if (h, w) == tuple(real) else f"MISMATCH (real {tuple(real)})"
    print(f"   {name:<8} {tuple(t.shape)} -> grid {h}x{w} -> {h*w:4d} tokens   [{ok}]")

> **Gotcha worth memorizing.** The downsampling is `ceil(size / 32)`, not `size // 32`. For a width of 1066 that's **34**, not 33 — a whole extra column of tokens. Every strided conv in ResNet uses padding that rounds up. If you ever hand-compute a feature-map size and the model disagrees by one, this is almost always why. (The cell above checks itself against the real backbone so you can trust the numbers.)

Variable sequence length is fine for a transformer (unlike a CNN feeding a fixed `nn.Linear`) — attention works on any length. This is precisely why DETR can skip the fixed-size crop.

**Cost warning.** Self-attention is O(L²) in sequence length. 850 tokens → an 850×850 attention matrix *per head, per layer*. That's the reason DETR uses the 32×-downsampled `layer4` map rather than something higher-resolution, and it's the reason the paper reports weak small-object AP (§4.1, `AP_S` 20.5 vs Faster R-CNN's 24.2).

## 5. Positional encodings: attention is blind to location

Self-attention is **permutation-invariant** — shuffle the 850 tokens and you get the same 850 outputs, just shuffled. So the raw sequence carries *no* information about where each token sat in the image. For detection, where things are is the entire point.

The fix: add a vector to each token that encodes its (row, col). DETR uses a 2-D extension of the sine encoding from *Attention Is All You Need*.

In [ ]:
with torch.no_grad():
    pos_embed = model.backbone[1](src, mask)   # the PositionEmbeddingSine defined in section 0

print("pos shape:", tuple(pos_embed.shape), "  <- same (B, 256, H, W) as the features")
print()
print("128 channels encode the row (y), 128 encode the column (x) -> 256 total")
print("see PositionEmbeddingSine.forward in the setup section")

In [ ]:
p = pos_embed[0].cpu()                       # (C, H, W) = (256, H, W), C = d_model

fig, axes = plt.subplots(2, 4, figsize=(13, 5))
for j, ch in enumerate([0, 1, 20, 60]):
    axes[0, j].imshow(p[ch]);       axes[0, j].set_title(f"ch {ch}  (y / row)", fontsize=9)
    axes[1, j].imshow(p[128 + ch]); axes[1, j].set_title(f"ch {128+ch}  (x / col)", fontsize=9)
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("Sine positional encoding: low channels = coarse position, high = fine")
plt.tight_layout(); plt.show()

Top row varies only **vertically** (it encodes the row), bottom row only **horizontally** (the column). Low channels are slow, smooth gradients; higher channels oscillate faster. Together, the 256 numbers at a grid cell form a unique fingerprint of its (x, y) — the same trick as binary place-value, but continuous.

### A subtlety: positions are *added at every attention layer*, not once at the input

In the original transformer you add the positional encoding to the input embedding once. DETR instead re-adds it to the **query and key** (but *not* the value) inside every attention layer:

In [ ]:
# The line that matters, from TransformerEncoderLayer.forward in section 0:
#
#     q = k = src + pos
#     src2 = self.self_attn(q, k, value=src)[0]
#
# pos is added to the QUERY and the KEY, but never to the VALUE. Position decides
# *where* to look; it is not part of what gets carried back.

layer = model.transformer.encoder.layers[0]
src_seq = proj.flatten(2).permute(2, 0, 1)          # (S, B, C)
pos_seq = pos_embed.flatten(2).permute(2, 0, 1)     # (S, B, C)
mask_seq = mask.flatten(1)                          # (B, S)

with torch.no_grad():
    out_with = layer(src_seq, mask_seq, pos_seq)
    out_zero = layer(src_seq, mask_seq, torch.zeros_like(pos_seq))
    out_shuf = layer(src_seq, mask_seq, pos_seq[torch.randperm(pos_seq.shape[0])])

print("one encoder layer, same features, different positional encoding:")
print(f"  pos as usual vs pos = 0        mean |diff| = {(out_with-out_zero).abs().mean():.4f}")
print(f"  pos as usual vs pos shuffled   mean |diff| = {(out_with-out_shuf).abs().mean():.4f}")
print()
print("Same pixels, different answer -- so position really is doing work.")
print("Re-run the layer with pos added to `value` as well and the model breaks:")
print("the checkpoint was never trained that way.")

Note `q = k = self.with_pos_embed(src, pos)` but `value=src`. Position influences *where to look* (the attention weights) but is not mixed into *what is retrieved* (the content).

The paper ablates this in **Table 3**: passing sine encodings at every attention layer scores **40.6 AP**; passing them once at the input drops to 39.2; removing spatial encodings entirely collapses to 32.8 (−7.8 AP). Position matters enormously.

## 6. The full journey, verified end to end

In [ ]:
with torch.no_grad():
    s, m = model.backbone[0](single.tensors, single.mask)
    pe_2d = model.backbone[1](s, m)
    pj = model.input_proj(s)
    sq = pj.flatten(2).permute(2, 0, 1)
    pe = pe_2d.flatten(2).permute(2, 0, 1)

rows = [
    ("PIL image",              f"(W={im.size[0]}, H={im.size[1]})"),
    ("after transform",        tuple(default_transform(im).shape)),
    ("batched (NestedTensor)", tuple(single.tensors.shape)),
    ("  + padding mask",       tuple(single.mask.shape)),
    ("ResNet-50 layer4",       tuple(s.shape)),
    ("  + downsampled mask",   tuple(m.shape)),
    ("input_proj (1x1 conv)",  tuple(pj.shape)),
    ("flattened sequence",     tuple(sq.shape)),
    ("positional encoding",    tuple(pe.shape)),
]
print(f"{'stage':<24} shape")
print("-" * 52)
for k, v in rows:
    print(f"{k:<24} {v}")
print()
print("The last two are what the transformer encoder actually receives.")

## What's next

`04` picks up exactly here: it feeds this `(850, 1, 256)` sequence into the encoder, then introduces the **object queries** that turn encoder memory into 100 predictions.

## Exercises

**Exercise 1.** Predict the feature-grid size for a 500×700 image *before* running it, then check.

<details><summary>Solution</summary>

Shortest side → 800, so 500×700 becomes 800×1120. Then `ceil(800/32) × ceil(1120/32)` = **25 × 35 = 875 tokens**.

```python
from PIL import Image
import math
fake = Image.new("RGB", (700, 500))            # PIL is (W, H)
t = default_transform(fake)
print("tensor:", tuple(t.shape))
print("predicted grid:", math.ceil(t.shape[1]/32), "x", math.ceil(t.shape[2]/32))
with torch.no_grad():
    nt = nested_tensor_from_tensor_list([t]).to(device)
    print("actual grid   :", tuple(model.backbone(nt)[0][-1].tensors.shape[-2:]))
```

Remember: `ceil`, not `//`.
</details>

---

**Exercise 2.** The paper's **DETR-DC5** removes the stride from ResNet's last stage, doubling feature resolution. Build it with `build_detr(dilation=True)` and measure the cost.

<details><summary>Solution</summary>

```python
dc5 = build_detr(dilation=True)
with torch.no_grad():
    f, _ = dc5.backbone(single.to("cpu"))
h, w = f[-1].tensors.shape[-2:]
print(f"DC5 grid {h}x{w} = {h*w} tokens (vs 25x34 = 850)")
print(f"-> {h*w/850:.1f}x tokens, {(h*w/850)**2:.0f}x attention cost")
```

You should get **50 × 67 = 3350 tokens** — 3.9× the tokens and about **16× the self-attention cost**, because attention is O(L²).

That is exactly the trade in Table 1: DETR-DC5 lifts small-object AP from 20.5 to 22.5 (**+2.0 AP_S**) and overall AP from 42.0 to 43.3, but GFLOPs go from 86 to 187 and FPS drops from 28 to 12 -- less than half the speed.
</details>

---

**Exercise 3.** Plot positional-encoding channels 126 and 127. Why do they look almost constant?

<details><summary>Solution</summary>

```python
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].imshow(p[126]); ax[0].set_title("ch 126 (sin)")
ax[1].imshow(p[127]); ax[1].set_title("ch 127 (cos)")
for a in ax: a.axis("off")
plt.show()
print("ch126 range:", float(p[126].min()), float(p[126].max()))
print("ch127 range:", float(p[127].min()), float(p[127].max()))
```

The wavelength is `dim_t = 10000 ** (2*(i//2)/128)`. At `i = 126/127` that's ≈ **8660**, so the angle fed to sin/cos across the whole image is only ≈ 0.0007 radians. Therefore `sin(θ) ≈ 0` and `cos(θ) ≈ 1` everywhere — nearly flat.

These are the *coarsest* channels: they'd only vary meaningfully over an image thousands of pixels wide. Low channels give fine detail, high channels give broad position — like the low and high bits of a binary number.
</details>

---

**Exercise 4.** What breaks if you pass a batch of two differently-sized images *without* the padding mask?

<details><summary>Solution</summary>

Nothing crashes — and that's the danger. The model happily attends to the grey padding as if it were image content, so predictions on the smaller image degrade in a way that's easy to miss.

```python
b2 = nested_tensor_from_tensor_list([default_transform(im), default_transform(im2)]).to(device)
with torch.no_grad():
    good = model(b2)                                     # with mask
    bad = model(b2.tensors)                              # plain tensor -> mask is all-False
p_good = good["pred_logits"][0].softmax(-1)[:, :-1].max(-1).values
p_bad = bad["pred_logits"][0].softmax(-1)[:, :-1].max(-1).values
print("detections with mask   :", (p_good > 0.9).sum().item())
print("detections without mask:", (p_bad > 0.9).sum().item())
```

This is why [`util/misc.py`](../util/misc.py)'s `collate_fn` exists and why you must pass it to your `DataLoader` (notebook `07` §8).
</details>